In [1]:
import sys
sys.path.append("../modules")

In [2]:
import torch
import torch.optim as optim
import pandas as pd
from pathlib import Path
from tqdm import tqdm

from custom_helpers.config import *
from custom_helpers.data_loading import chunk_df, build_loader, construct_fixed_region
from custom_helpers.sample_generation import  generate_samples, load_all_samples
from custom_helpers.scoring import load_scoring_model, construct_mutated_dataset, score_mutated_dataset

from ProteinMPNN.protein_mpnn_utils import tied_featurize, ProteinMPNN as ProteinMPNNVal
from ProteinMPNN.training.model_utils import loss_smoothed, loss_nll, get_std_opt, ProteinMPNN as ProteinMPNNTrain

/home/ghandill/miniconda3/envs/fresh_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
class MyArgs(object):
  def __init__(self):
    self.path_for_training_data = "../../data/ppb_affinity"
    self.path_for_outputs = "./test"
    self.previous_checkpoint = ""
    self.num_epochs = 30
    self.save_model_every_n_epochs = 5
    self.reload_data_every_n_epochs = 4
    self.num_examples_per_epoch = 200
    self.total_seq_length = 10000
    self.train_batch_size = 64
    self.val_batch_size = 5
    self.max_protein_length = 2000
    self.hidden_dim = 128
    self.num_encoder_layers = 3
    self.num_decoder_layers = 3
    self.num_neighbors = 32
    self.dropout = 0.1
    self.backbone_noise = 0.1
    self.rescut = 3.5
    self.debug = False
    self.gradient_norm = -1.0 #no norm
    self.mixed_precision= True 
    self.val_batch_copies = 5
    self.sample_temp = 1
    self.lambda_reward = 0.25
    self.beta_dpo = 0.1

main_args = MyArgs()

scoring_args = {
    "checkpoint_path": "/home/ghandill/masterthesis_lilit_ghandilyan/modules/PPB-Affinity-HF/checkpoints/checkpoint.pt",
    "device": torch.device("cuda:1" if (torch.cuda.is_available()) else "cpu"),
}

In [4]:
def chunk_df_by_pdb(df, chunk_size):
    # Group by PDB column
    grouped = df.groupby('pdb')
    
    current_chunk = []
    current_size = 0
    
    for pdb, group in grouped:
        group_size = len(group)
        
        # If this PDB group is larger than chunk_size, split it into its own chunks
        if group_size >= chunk_size:
            # First, yield any accumulated chunk
            if current_chunk:
                yield pd.concat(current_chunk, ignore_index=True)
                current_chunk = []
                current_size = 0
            
            # Then split this large PDB group into multiple chunks
            for i in range(0, group_size, chunk_size):
                yield group.iloc[i:i+chunk_size]
        
        # If adding this group exceeds chunk_size, yield current chunk first
        elif current_size + group_size > chunk_size:
            if current_chunk:
                yield pd.concat(current_chunk, ignore_index=True)
            current_chunk = [group]
            current_size = group_size
        
        # Otherwise, add to current chunk
        else:
            current_chunk.append(group)
            current_size += group_size
    
    # Yield the last chunk if it exists
    if current_chunk:
        yield pd.concat(current_chunk, ignore_index=True)

In [5]:
split_type = "medium_split"

data_path_base = Path.cwd().parent / "data"
affinity_df = pd.read_csv(data_path_base / 'processed_data.csv', index_col=0)

train_set_df = affinity_df[affinity_df[split_type] == "train"]
val_set_df = affinity_df[affinity_df[split_type] == "val"]

train_chunks = list(chunk_df_by_pdb(train_set_df, main_args.train_batch_size))

In [6]:
train_model_device = torch.device("cuda:3" if (torch.cuda.is_available()) else "cpu")
val_model_device = torch.device("cuda:7" if (torch.cuda.is_available()) else "cpu")
ref_model_device = torch.device("cuda:4" if (torch.cuda.is_available()) else "cpu")

train_model = ProteinMPNNTrain(node_features=main_args.hidden_dim, 
                    edge_features=main_args.hidden_dim, 
                    hidden_dim=main_args.hidden_dim, 
                    num_encoder_layers=main_args.num_encoder_layers, 
                    num_decoder_layers=main_args.num_encoder_layers, 
                    k_neighbors=main_args.num_neighbors, 
                    dropout=main_args.dropout, 
                    augment_eps=main_args.backbone_noise,
                    num_letters=21)
train_model.to(train_model_device)
checkpoint_path = "../modules/ProteinMPNN/vanilla_model_weights/v_48_002.pt"
checkpoint = torch.load(checkpoint_path, map_location=train_model_device)

train_model.load_state_dict(checkpoint['model_state_dict'])

val_model = ProteinMPNNVal(node_features=main_args.hidden_dim, 
                    edge_features=main_args.hidden_dim, 
                    hidden_dim=main_args.hidden_dim, 
                    num_encoder_layers=main_args.num_encoder_layers, 
                    num_decoder_layers=main_args.num_encoder_layers, 
                    k_neighbors=main_args.num_neighbors, 
                    dropout=main_args.dropout, 
                    augment_eps=main_args.backbone_noise,
                    num_letters=21)
val_model.to(val_model_device)

for param in val_model.parameters():
    param.requires_grad = False
    
val_model.load_state_dict(train_model.state_dict())

ref_model = ProteinMPNNTrain(node_features=main_args.hidden_dim, 
                    edge_features=main_args.hidden_dim, 
                    hidden_dim=main_args.hidden_dim, 
                    num_encoder_layers=main_args.num_encoder_layers, 
                    num_decoder_layers=main_args.num_encoder_layers, 
                    k_neighbors=main_args.num_neighbors, 
                    dropout=main_args.dropout, 
                    augment_eps=main_args.backbone_noise,
                    num_letters=21)
ref_model.to(ref_model_device)

for param in ref_model.parameters():
    param.requires_grad = False
    
ref_model.load_state_dict(train_model.state_dict())

<All keys matched successfully>

In [7]:
import wandb
from datetime import datetime

wandb_log = False

if wandb_log:
    current_time = datetime.now().strftime("%Y-%m-%d_%H-%M")
    wandb.init(
        project="thesis",
        config=main_args.__dict__,
        name=f"time_split_dpo_{current_time}",
    )


In [8]:
import pickle


# def precompute_fixed_positions(df):
#     fixed_positions_dict = {}
#     # Wrap the iterator with tqdm
#     for i, row in tqdm(df.iterrows(), total=len(df), desc="Precomputing fixed positions"):
#         row_id = row["complex_id"]
#         fixed_positions_dict[row_id] = construct_fixed_region(row)
#     return fixed_positions_dict

# # Run once
# fixed_positions_cache = precompute_fixed_positions(affinity_df)

# # Save
# with open("fixed_positions_cache.pkl", "wb") as f:
#     pickle.dump(fixed_positions_cache, f)

# Load
with open("cache/fixed_positions_cache.pkl", "rb") as f:
    fixed_positions_cache = pickle.load(f)


In [9]:
total_step = 0
scaler = torch.amp.GradScaler()

optimizer = optim.Adam(
    train_model.parameters(),
    lr=5e-5  # good default for fine-tuning transformer-type models
)

def train_epoch_dpo(train_model, reference_model, args, epoch, train_device):
    print(f"\n=== Epoch {epoch+1}/{args.num_epochs} ===")
    train_model.train()

    # Initialize at the top
    dpo_loss_sum = 0.0
    dpo_count = 0
    num_pairs_total = 0

    for _, chunk in tqdm(enumerate(train_chunks), total=len(train_chunks), desc="Train chunks"):
        loader_train, chain_id_dict_train, fixed_positions_dict, _ = build_loader(chunk, args.total_seq_length, fixed_positions_cache)
        for batch in loader_train:
            X, S, mask, _, chain_M, chain_encoding_all, _, visible_list_list, masked_list_list, masked_chain_length_list_list, chain_M_pos, omit_AA_mask, residue_idx, _, _, pssm_coef, pssm_bias, pssm_log_odds_all, bias_by_res_all, _ = tied_featurize(
                batch, train_device, chain_id_dict_train, fixed_positions_dict,
                omit_AA_dict, tied_positions_dict, pssm_dict, bias_by_res_dict
            )
            optimizer.zero_grad()
            mask_for_loss = mask * chain_M * chain_M_pos
            
            rewards = torch.tensor([-el["score"] for el in batch], device=train_device)
            pdb_ids = [el["pdb"] for el in batch]
            
            batch_size = len(batch)
            
            if args.mixed_precision:
                with torch.amp.autocast(device_type="cuda"):
                    log_probs = train_model(X, S, mask, chain_M, residue_idx, chain_encoding_all)
                    loss_tokenwise, _ = loss_smoothed(S, log_probs, mask_for_loss)
                    
                    with torch.no_grad():
                        ref_log_probs = reference_model(X, S, mask, chain_M, residue_idx, chain_encoding_all)
                        ref_loss_tokenwise, _ = loss_smoothed(S, log_probs, mask_for_loss)
                    
                     # Compute per-sequence log probabilities
                    seq_loss = torch.sum(loss_tokenwise * mask_for_loss, dim=-1) / (mask_for_loss.sum(dim=-1) + 1e-8)
                    ref_seq_loss = torch.sum(loss_tokenwise * mask_for_loss, dim=-1) / (mask_for_loss.sum(dim=-1) + 1e-8)
                
                # DPO loss: only for pairs from the SAME PDB
                loss_dpo = 0
                num_pairs = 0
                
                for i in range(batch_size):
                    for j in range(i + 1, batch_size):
                        # CRITICAL: Only compare sequences from the same PDB
                        if pdb_ids[i] != pdb_ids[j]:
                            continue
                        
                        if rewards[i] > rewards[j]:
                            # i is preferred over j
                            preferred_logprob = seq_loss[i]
                            dispreferred_logprob = seq_loss[j]
                            preferred_ref = ref_seq_loss[i]
                            dispreferred_ref = ref_seq_loss[j]
                        elif rewards[j] > rewards[i]:
                            # j is preferred over i
                            preferred_logprob = seq_loss[j]
                            dispreferred_logprob = seq_loss[i]
                            preferred_ref = ref_seq_loss[j]
                            dispreferred_ref = ref_seq_loss[i]
                        else:
                            # Equal rewards, skip this pair
                            continue
                        
                        # DPO loss: -log(sigmoid(beta * (log_ratio_preferred - log_ratio_dispreferred)))
                        log_ratio_preferred = preferred_logprob - preferred_ref
                        log_ratio_dispreferred = dispreferred_logprob - dispreferred_ref
                        
                        loss_dpo += -torch.nn.functional.logsigmoid(
                            args.beta_dpo * (log_ratio_preferred - log_ratio_dispreferred)
                        )
                        num_pairs += 1
                
                if num_pairs > 0:
                    loss_dpo = loss_dpo / num_pairs
                    
                    # Track statistics
                    dpo_loss_sum += loss_dpo.item()
                    dpo_count += 1
                    num_pairs_total += num_pairs
                else:
                    # Skip update if no valid pairs in this batch
                    continue
                    
            scaler.scale(loss_dpo).backward()
            
            if args.gradient_norm > 0.0:
                total_norm = torch.nn.utils.clip_grad_norm_(train_model.parameters(), args.gradient_norm)
            
            scaler.step(optimizer)
            scaler.update()

            # still compute these metrics exactly as before
            # loss, loss_av, true_false = loss_nll(S, log_probs, mask_for_loss)
            # train_sum     += torch.sum(loss * mask_for_loss).cpu().data.numpy()
            # train_acc     += torch.sum(true_false * mask_for_loss).cpu().data.numpy()
            # train_weights += torch.sum(mask_for_loss).cpu().data.numpy()
            
            avg_dpo_loss = dpo_loss_sum / max(dpo_count, 1)
    return train_model, avg_dpo_loss

In [10]:
def score_subset(scoring_model, subset_df):
    subset_chunks = list(chunk_df(subset_df, main_args.val_batch_size // main_args.val_batch_copies))
    
    all_samples = []

    for _, chunk in tqdm(enumerate(subset_chunks), total=len(subset_chunks), desc="Scoring a subset"):
        samples, chunk_loss = generate_samples(val_model,
                                chunk,
                                device=val_model_device,
                                fixed_positions_cache=fixed_positions_cache,
                                args=main_args)
        
        all_samples.extend(samples)
    mutated_df = construct_mutated_dataset(subset_df, all_samples)
    final_results = score_mutated_dataset(mutated_df, scoring_model, scoring_args)
    return final_results.score.mean(), chunk_loss

In [11]:
train_loss = None
scoring_model = load_scoring_model(scoring_args)
train_subset = train_set_df.sample(32)
val_subset = val_set_df.sample(32)

for epoch in range(main_args.num_epochs):
    train_model, train_loss = train_epoch_dpo(train_model, ref_model, main_args, epoch, train_model_device)
    val_model.load_state_dict(train_model.state_dict())
    
    train_score, train_val_loss = score_subset(scoring_model, train_subset)
    val_score, val_val_loss = score_subset(scoring_model, train_subset)
    
    wandb.log({"Train Loss": train_loss, 
               "Train Score": train_score,
               "Vaidation Score": val_score,
               "Validation on train loss": train_val_loss,
               "Validation loss": val_val_loss})



=== Epoch 1/30 ===


Train chunks:   0%|          | 0/197 [00:01<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 42.00 MiB. GPU 3 has a total capacity of 10.75 GiB of which 30.69 MiB is free. Process 4113847 has 10.11 GiB memory in use. Including non-PyTorch memory, this process has 616.00 MiB memory in use. Of the allocated memory 345.64 MiB is allocated by PyTorch, and 92.36 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)